In [1]:
import tensorflow as tf
import numpy as np
from qkeras.utils import _add_supported_quantized_objects

FILE_PATH = '/home/aya/HLS4ML_VS_MANUAL/src/hdl/RHEED_Gaussian/Models/gaussian_nov24.keras'

def dice_loss(y_true, y_pred, delta=0.5):

    weights = tf.constant([3.0, 1.0, 2.0, 2.0, 0.1], dtype=tf.float32)
    y_pred = y_pred + 1e-8
    diff = y_true - y_pred
    abs_diff = tf.abs(diff)

    quadratic = tf.minimum(abs_diff, delta)
    linear = abs_diff - quadratic
    huber_loss = 0.5 * tf.square(quadratic) + delta * linear

    weighted_loss = weights * huber_loss
    return tf.reduce_mean(weighted_loss)
    
co = {}
_add_supported_quantized_objects(co)
# Add any custom losses
co["dice_loss"] = dice_loss

model = tf.keras.models.load_model(FILE_PATH, custom_objects=co)
model.summary()

2026-05-14 14:11:31.286127: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-14 14:11:34.868563: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-14 14:11:35.769166: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-05-14 14:11:35.769188: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore 

2026-05-14 14:11:48.992600: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2026-05-14 14:11:48.992717: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory
2026-05-14 14:11:48.992725: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries. If you would like to use Nvidia GPU with TensorRT, please make sure the missing libraries mentioned above are installed properly.
2026-05-14 14:12:15.750648: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-0

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


Model: "pp_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 q_conv2d_batchnorm (QConv2D  (None, 46, 46, 6)        85        
 Batchnorm)                                                      
                                                                 
 q_activation (QActivation)  (None, 46, 46, 6)         0         
                                                                 
 max_pooling2d (MaxPooling2D  (None, 11, 11, 6)        0         
 )                                                               
                                                                 
 q_conv2d_batchnorm_1 (QConv  (None, 9, 9, 8)          473       
 2DBatchnorm)                                                    
                                                                 
 q_activation_1 (QActivation  (None, 9, 9, 8)          0         
 )                                                        

In [2]:
layer = model.get_layer("q_conv2d_batchnorm")


# weights = layer.get_folded_weights() # batchnorm params already folded into conv weights
# weights = layer.get_weights()

# print([w.shape for w in layer.get_folded_weights()])
# print([w.shape for w in layer.get_weights()])

for layer in model.layers:
    if "conv" in layer.name:
        print([w.shape for w in layer.get_folded_weights()])
    else:
        print([w.shape for w in layer.get_weights()])
# print(weights)

# weights are float32 at this point
# fused_weights, fused_bias = layer.get_folded_weights()
# print(fused_weights)
# print(fused_bias)


[TensorShape([3, 3, 1, 6]), TensorShape([6])]
[]
[]
[TensorShape([3, 3, 6, 8]), TensorShape([8])]
[]
[]
[TensorShape([3, 3, 8, 10]), TensorShape([10])]
[]
[]
[]
[(10, 15), (15,)]
[]
[(15, 10), (10,)]
[]
[(10, 5), (5,)]


In [1]:
import numpy as np
import os

def dec_to_bin(number : int | float, bits=-1):
    """
    Converts a decimal number to a str binary representation
    
    :param number: Number to convert into binary
    :type number: int | float
    :param bits: Bitwidth of output number. If negative, uses the minimum amount
    """
    # Determines if a number is negative
    neg=False
    if (number<0):
        number*=-1
        number-=1
        neg=True
    
    # Rounds number to nearest int
    number=int(np.round(number, 0))
    out=""

    # Return maximum positive or negative number if the input cannot be represented by the bitwidth
    if (bits>0 and number>2**(bits-1)):
        return "0" + "1"*(bits-1)
    elif (bits>0 and number<(-1)*2**(bits-1)):
        return "1" + "0"*(bits-1)
    
    # Do typical conversion of decimal to binary number
    while (number>0):
        res = number%2
        if (neg):
            res= 0 if (res==1) else 1
        out=f"{res}{out}"
        if (len(out)==(bits-1)):
            break
        number=int(number/2)
        
    # Add signed bit
    if (neg):
        out=f"{1}{out}"
    else: out=f"{0}{out}"

    # If the number was 0
    if (len(out)==0):
        out="0"

    # Sign extension
    while (len(out)<bits):
        out=f"{out[0]}{out}"

    return out

In [4]:
def gen_weight(accuracy, model, target_dir="./"):
    """
    Generate weight and bias packages from keras model
    :param accuracy: Accuracy for packages. Formatted (width, integers)
    :type accuracy: (int, int)

    """
    # The start of each number
    head = f"{accuracy[0]}'b"

    Nfrac = accuracy[0] - accuracy[1]
    # The amount of weights for each layer
    current = 0
    for layer in model.layers:
        if 'conv2d_batchnorm' in layer.name.lower():
            # fused_weights shape: 4Darray (height, width, inputChannels, outputChannels)
            # fused_bias: 1D array
            fused_weights, fused_bias = layer.get_folded_weights()
            # transpose to match sv indexing in conv2d module: (o, i, h, w)
            fused_weights_t = np.transpose(fused_weights,(3, 2, 0, 1))
            # flatten
            fused_weights_f = fused_weights_t.reshape(-1)
            contents = [fused_weights_f, fused_bias]
            # debug
            print(type(fused_weights_f))
            print(fused_weights_f.shape)
            print(type(fused_weights_f[0]))
            print(fused_weights_f[0].shape)
        else:
            contents = layer.get_weights()

        # need to also check if layer.get_weights() returns empty list and skip (maxpool)
        if (contents!=None and len(contents) > 0): # for dense layers
            # contents = np.concatenate((contents, np.zeros(len(contents[1]))), axis=1)
            if 'conv2d_batchnorm' in layer.name.lower():
                weights = [fused_weights_f]
                biases  = [fused_bias]
            else:
                weights = []
                biases = []
                for cont in contents:
                    try:
                        len(cont[0])
                        weights.append(cont)
                    except:
                        biases.append(cont)
                if (len(weights)!=len(biases)):
                    print(len(weights), len(biases))
                    biases.append(np.zeros_like(biases[0]))
            for i in range(len(weights)):
                name = layer.name
                # Converting the files to arrays
                weight = weights[i]
                bias = biases[i]

                # Output file name
                filename = os.path.join(target_dir, f"{name}_pkg_{accuracy[0]}_{accuracy[1]}_{i}.sv")
                # if (not os.path.isfile(filename)):
                with open(filename, "w") as f:
                    # Writes the header to the file
                    f.write(f"//Width: {accuracy[0]}\n//Int: {accuracy[1]}\n")
                    f.write(f"package {name}_{i}_{accuracy[0]}_{accuracy[1]};\n\n")

                    ### NEW for 2d convolution ### (since weights are now flattened instead of 2d like dense)
                    if 'conv2d_batchnorm' in name.lower():
                        f.write(f"localparam logic signed [{accuracy[0]-1}:0] convWeights [0:{len(weight)-1}] = '" + "{\n")

                        # Writes the main body of the function
                        for i in range(len(weight)):
                            # f.write("{")
                            num = dec_to_bin(weight[i]*(2**(Nfrac)), accuracy[0])
                            f.write(f"{head}{num}")
                            if (i!=len(weight)-1): 
                                f.write(",\n")
                        f.write("};\n")
                        f.write(f"localparam logic signed [{accuracy[0]-1}:0] convBiases [{len(bias)}] = '"+"{\n")
                        for i in range(0, len(bias)):
                            num = dec_to_bin(bias[i]*(2**Nfrac), accuracy[0])
                            f.write(f"{head}{num}")
                            f.write(",\n" if i!=(len(bias)-1) else "\n};\nendpackage")
                    #############################
                    else:
                        f.write(f"localparam logic signed [{accuracy[0]-1}:0] weights [{len(weight)}][{len(weight[0])}] = '" + "{\n")

                        # Writes the main body of the function
                        for i in range(len(weight)):
                            f.write("{")
                            num = dec_to_bin(weight[i][0]*(2**(Nfrac)), accuracy[0])
                            f.write(f"{head}{num}")
                            for j in range(1, len(weight[0])):
                                num = dec_to_bin(weight[i][j]*(2**(Nfrac)), accuracy[0])
                                f.write(f", {head}{num}")
                            if (i!=len(weight)-1): 
                                f.write("},\n")
                        f.write("}\n};\n")
                                                                                # len(weights[0]) happens to be len(bias) for dense layers
                        f.write(f"localparam logic signed [{accuracy[0]-1}:0] bias [{len(weight[0])}] = '"+"{\n")
                        for i in range(0, len(bias)):
                            num = dec_to_bin(bias[i]*(2**Nfrac), accuracy[0])
                            f.write(f"{head}{num}")
                            f.write(",\n" if i!=(len(bias)-1) else "\n};\nendpackage")

In [5]:
TOTAL_BITS = 16
NUM_INT = 6

gen_weight(accuracy=(TOTAL_BITS, NUM_INT), model=model, target_dir="./")


<class 'numpy.ndarray'>
(54,)
<class 'numpy.float32'>
()
<class 'numpy.ndarray'>
(432,)
<class 'numpy.float32'>
()
<class 'numpy.ndarray'>
(720,)
<class 'numpy.float32'>
()


In [2]:
def gen_conv2d_test_data(accuracy, input_width, num_filters, filt_dimension, target_dir="./"):
    """
    Generate test data package for conv2d_reuse9 testbench
    :param accuracy: Accuracy for packages. Formatted (width, integers)
    :type accuracy: (int, int)

    """
    # The start of each number
    head = f"{accuracy[0]}'b"

    # Nfrac = accuracy[0] - accuracy[1]

    # Generates random weights, biases, and input pixels
    num_weights = num_filters * filt_dimension**2
    # np.random.randint(low, high, size)
    # range of signed two's complement: -2^(n-1) to +2^(n-1)-1
    weights = np.random.randint(-(2**(accuracy[0]-1)), 2**(accuracy[0]-1), num_weights).tolist()
    biases   = np.random.randint(-(2**(accuracy[0]-1)), 2**(accuracy[0]-1), num_filters).tolist()
    pixels  = np.random.randint(-(2**(accuracy[0]-1)), 2**(accuracy[0]-1), input_width**2).tolist()
    
    # Output file name
    filename = os.path.join(target_dir, 
                            f"conv2d_reuse9_test_data_{accuracy[0]}_{accuracy[1]}_{input_width}_{num_filters}_{filt_dimension}.sv")

    with open(filename, "w") as f:
        # Writes the header to the file
        f.write(f"package conv2d_reuse9_test_data;\n")

        # inputMatrix (1D array)
        f.write(f"localparam logic signed [{accuracy[0]-1}:0] inputMatrix [0:{len(pixels)-1}] = '" + "{\n")
        for i in range(len(pixels)):
            # don't need to multiply by 2**Nfrac because randint already gives us integers, not floating point
            num = dec_to_bin(pixels[i], accuracy[0])
            f.write(f"{head}{num}")
            if (i!=len(pixels)-1): 
                f.write(",\n")
        f.write("};\n")
        # convWeights
        f.write(f"localparam logic signed [{accuracy[0]-1}:0] convWeights [0:{len(weights)-1}] = '" + "{\n")
        for i in range(len(weights)):
            num = dec_to_bin(weights[i], accuracy[0])
            f.write(f"{head}{num}")
            if (i!=len(weights)-1): 
                f.write(",\n")
        f.write("};\n")
        # convBiases
        f.write(f"localparam logic signed [{accuracy[0]-1}:0] convBiases [{len(biases)}] = '"+"{\n")
        for i in range(0, len(biases)):
            num = dec_to_bin(biases[i], accuracy[0])
            f.write(f"{head}{num}")
            f.write(",\n" if i!=(len(biases)-1) else "\n};\nendpackage")

In [4]:
import numpy as np
import os

TOTAL_BITS = 16
NUM_INT = 6
INPUT_WIDTH = 501
NUM_FILTERS = 2
FILT_DIMENSION = 3

gen_conv2d_test_data(accuracy=(TOTAL_BITS, NUM_INT), 
                     input_width=INPUT_WIDTH, 
                     num_filters=NUM_FILTERS, 
                     filt_dimension=FILT_DIMENSION, 
                     target_dir="./")